In [ ]:
import os
import sys
main_dir = os.path.dirname(os.getcwd())
sys.path.append(main_dir)

from sklearn.decomposition import PCA
from matplotlib.colors import LogNorm
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt


def define_model(num_neurons, num_depth):
    """Build a simple MLP with `num_depth` hidden ReLU layers of `num_neurons` each."""
    layers = [nn.Linear(1, num_neurons), nn.ReLU()]
    for _ in range(num_depth - 1):
        layers.extend([nn.Linear(num_neurons, num_neurons), nn.ReLU()])
    layers.append(nn.Linear(num_neurons, 1))
    return nn.Sequential(*layers)


def snapshot_weights(model):
    """Flatten all parameters into a single 1D vector."""
    return np.concatenate([p.detach().cpu().numpy().flatten() for p in model.parameters()])

from src.utils import degree_3_polynomial
A = 1 
B = 2 
C = -0.5 
D = 0.04 
x_true = np.linspace(0, 10, 1000)
y_true = degree_3_polynomial(x_true, coeffs=[A, B, C, D], noise_std=0.8)

sample_size = 20
sampling_indices = np.random.choice(len(x_true), size=sample_size, replace=False)
x_sample = x_true[sampling_indices]
y_sample = y_true[sampling_indices] 
# ─── Standardize data once (same for all runs) ───
x_sample_mean, x_sample_std = x_sample.mean(), x_sample.std()
y_sample_mean, y_sample_std = y_sample.mean(), y_sample.std()
x_test = np.linspace(-5, 15, 1000)
y_test = degree_3_polynomial(x_test, coeffs=[A, B, C, D], noise_std=0)
x_train_std = (x_sample - x_sample_mean) / x_sample_std
y_train_std = (y_sample - y_sample_mean) / y_sample_std
x_train_tensor = torch.from_numpy(x_train_std).float().unsqueeze(1)
y_train_tensor = torch.from_numpy(y_train_std).float().unsqueeze(1)




In [ ]:
# ─── Run training N times with different random inits ───
n_runs = 50
n_epochs = 2000
snapshot_every = 100
num_neurons = 32       # change to 10, 32, etc. to see how capacity changes the picture
num_depth = 1

all_run_histories = []    # weight trajectories
all_run_losses = []       # loss at EACH snapshot, one array per run
final_losses = []         # loss at end of each run
final_state_dicts = []    # trained weights for evaluation

for run in range(n_runs):
    torch.manual_seed(run)                               # different init per run
    model = define_model(num_neurons=num_neurons, num_depth=num_depth)
    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
    
    # Initial state and initial loss
    run_hist = [snapshot_weights(model)]
    with torch.no_grad():
        run_losses = [criterion(model(x_train_tensor), y_train_tensor).item()]
    
    for epoch in range(n_epochs):
        optimizer.zero_grad()
        loss = criterion(model(x_train_tensor), y_train_tensor)
        loss.backward()
        optimizer.step()
        if (epoch + 1) % snapshot_every == 0:
            run_hist.append(snapshot_weights(model))
            run_losses.append(loss.item())
    
    all_run_histories.append(np.array(run_hist))
    all_run_losses.append(np.array(run_losses))
    final_losses.append(loss.item())
    final_state_dicts.append({k: v.clone() for k, v in model.state_dict().items()})

all_run_histories = np.array(all_run_histories)
all_run_losses = np.array(all_run_losses)
final_losses = np.array(final_losses)

print(f"Ran {n_runs} training runs with {num_neurons} hidden neurons.")
print(f"Final losses: min={final_losses.min():.4f}, "
      f"median={np.median(final_losses):.4f}, "
      f"max={final_losses.max():.4f}")



In [ ]:

# ─── Fit PCA on all snapshots from all runs ───
all_snapshots = all_run_histories.reshape(-1, all_run_histories.shape[-1])
pca = PCA(n_components=2)
pca.fit(all_snapshots)
trajs = [pca.transform(hist) for hist in all_run_histories]

print(f"PC1 captures {pca.explained_variance_ratio_[0]*100:.1f}% of variance")
print(f"PC2 captures {pca.explained_variance_ratio_[1]*100:.1f}% of variance")



In [ ]:

# ─── Plot ───
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Shared color scale across all snapshots (log, since loss spans orders of magnitude)
all_losses_flat = all_run_losses.flatten()
cmap = plt.cm.plasma_r
norm = LogNorm(vmin=all_losses_flat.min(), vmax=all_losses_flat.max())


# ─── Left: weight-space trajectories, dots colored by snapshot loss ───
ax = axes[0]

for run_idx, traj in enumerate(trajs):
    run_losses = all_run_losses[run_idx]
    
    # Line segments, each colored by its starting snapshot's loss
    for i in range(len(traj) - 1):
        ax.plot(traj[i:i+2, 0], traj[i:i+2, 1], '-',
                color=cmap(norm(run_losses[i])),
                alpha=0.4, linewidth=1)
    
    # All snapshots as dots, colored by their loss
    ax.scatter(traj[:, 0], traj[:, 1],
               c=run_losses, cmap=cmap, norm=norm,
               s=10, alpha=0.5)
    
    # Distinct markers for start and end
    ax.scatter(traj[0, 0], traj[0, 1], marker='o',
               facecolor='none', edgecolor=cmap(norm(run_losses[0])),
               s=40, linewidth=1.5)
    ax.scatter(traj[-1, 0], traj[-1, 1], marker='*',
               color=cmap(norm(run_losses[-1])),
               s=100, edgecolor='black', linewidth=0.5)

sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
plt.colorbar(sm, ax=ax, label='Loss at each snapshot (log)')
ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% variance)')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% variance)')
ax.set_title(f'{n_runs} trajectories in weight space\n(○ start, ★ end; color = loss at that point)')
ax.grid(alpha=0.3)


# ─── Right: the functions each run learned, colored by final loss ───
ax = axes[1]

x_test_std = (x_test - x_sample_mean) / x_sample_std
x_test_tensor = torch.from_numpy(x_test_std).float().unsqueeze(1)

for run_idx in range(n_runs):
    model = define_model(num_neurons=num_neurons, num_depth=num_depth)
    model.load_state_dict(final_state_dicts[run_idx])
    model.eval()
    with torch.no_grad():
        y_pred_std = model(x_test_tensor).cpu().numpy().flatten()
    y_pred = y_pred_std * y_sample_std + y_sample_mean
    
    ax.plot(x_test, y_pred,
            color=cmap(norm(final_losses[run_idx])),
            alpha=0.4, linewidth=1)

ax.plot(x_test, y_test, color="black", label="True function",
        linestyle='--', linewidth=2)
ax.scatter(x_sample, y_sample, color="blue", s=30,
           label="Training data", zorder=5)
ax.set_xlabel('x')
ax.set_ylabel('y')
ax.set_ylim(-5, 30)
ax.set_xlim(-5, 15)
ax.set_title(f'Functions learned by the {n_runs} runs\n(color = final loss)')
ax.legend()
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# for one specific run plot the (a) trajectory in weight space (b) the function learned at each snapshot, colored by loss at that snapshot
run_idx = np.random.randint(n_runs)  # pick a random run to visualize
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
# ─── Left: weight-space trajectory for the selected run ───
ax = axes[0]
traj = trajs[run_idx]
run_losses = all_run_losses[run_idx]
for i in range(len(traj) - 1):
    ax.plot(traj[i:i+2, 0], traj[i:i+2, 1], '-',
            color=cmap(norm(run_losses[i])),
            alpha=0.4, linewidth=1)
ax.scatter(traj[:, 0], traj[:, 1],
              c=run_losses, cmap=cmap, norm=norm,
              s=10, alpha=0.5)
ax.scatter(traj[0, 0], traj[0, 1], marker='o',
               facecolor='none', edgecolor=cmap(norm(run_losses[0])),
               s=40, linewidth=1.5)
ax.scatter(traj[-1, 0], traj[-1, 1], marker='*',
                color=cmap(norm(run_losses[-1])),
                s=100, edgecolor='black', linewidth=0.5)
sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
plt.colorbar(sm, ax=ax, label='Loss at each snapshot (log)')
ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% variance)')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% variance)')
ax.set_title(f'Trajectory of run {run_idx} in weight space\n(○ start, ★ end; color = loss at that point)')
ax.grid(alpha=0.3)
# ─── Right: functions learned at each snapshot for the selected run ───
ax = axes[1]
model = define_model(num_neurons=num_neurons, num_depth=num_depth)
for snapshot_idx in range(all_run_histories.shape[1]):
    snapshot_weights_vector = all_run_histories[run_idx, snapshot_idx]
    # Reconstruct state_dict from the flattened weights
    state_dict = {}
    pointer = 0
    for name, param in model.state_dict().items():
        num_params = param.numel()
        state_dict[name] = torch.from_numpy(snapshot_weights_vector[pointer:pointer+num_params].reshape(param.shape)).float()
        pointer += num_params
    model.load_state_dict(state_dict)
    model.eval()
    with torch.no_grad():
        y_pred_std = model(x_test_tensor).cpu().numpy().flatten()
    y_pred = y_pred_std * y_sample_std + y_sample_mean
    ax.plot(x_test, y_pred,
            color=cmap(norm(run_losses[snapshot_idx])),
            alpha=0.4, linewidth=1)
ax.plot(x_test, y_test, color="black", label="True function", 
        linestyle='--', linewidth=2)
ax.scatter(x_sample, y_sample, color="blue", s=30, label="Training data", zorder=5)
ax.set_xlabel('x')
ax.set_ylabel('y')
ax.set_ylim(-5, 30)
ax.set_xlim(-5, 15)
ax.set_title(f'Functions learned at each snapshot for run {run_idx}\n(color = loss at that snapshot)')
ax.legend()
